# Notebook 03 Draft Scratchpad

## Hypothesis Testing and Reliability Analysis

This scratchpad notebook is used to prototype the statistical analyses planned for Notebook 03. It contains trial window definitions, candidate tests, intermediate outputs, and rough comparisons. Only analyses that prove useful and interpretable here will be moved into the final clean notebook.

### Analysis Roadmap

**Setup**
- imports, data loading, variable selections, configuration

**MetroPT-3: pre-failure vs normal comparisons**
- pre-failure window definition
- normal-operation baseline definition
- Mann-Whitney U tests on selected continuous sensors

**Hydraulic system: condition-based comparisons**
- Mann-Whitney U / Kruskal-Wallis tests on selected feature-condition pairs
- stable vs unstable cycle comparisons

**Reliability metrics**
- MTBF and MTTR for both datasets

**Cross-dataset interpretation notes**
- comparison of statistically supported findings across shared physical concepts

**Keep / drop decisions for final Notebook 03**
- which results move to the clean notebook, which stay in scratchpad

## Setup

This section loads the processed MetroPT-3 and hydraulic datasets, imports the statistical tools required for hypothesis testing, and defines the initial variable selections carried forward from Notebook 02.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Reproducibility and display
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_colwidth', None)

# Load processed datasets
metro = pd.read_csv('../data/processed/metro_cleaned.csv', parse_dates=['timestamp'])
hydraulics = pd.read_csv('../data/processed/hydraulics_features.csv')

# Load MetroPT-3 documented failure events
metro_failures = pd.read_csv('../data/raw/metropt3_failure_reports.csv', parse_dates=['start_time', 'end_time'], keep_default_na=False)
metro_failures['duration'] = metro_failures['end_time'] - metro_failures['start_time']

print(f"MetroPT-3 shape: {metro.shape}")
print(f"Hydraulic shape: {hydraulics.shape}")
print(f"\nMetroPT-3 failure events: {len(metro_failures)}")
print(metro_failures)

# Variable selections guided by Notebook 02
metro_vars = ['TP2', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current']

hydraulic_pairs = {
    'cooler_condition': ['TS1_mean', 'TS1_std'],
    'pump_leakage': ['EPS1_mean', 'PS1_mean'],
    'stable_flag': ['TS1_std', 'VS1_std', 'PS1_std', 'EPS1_std'],
}

MetroPT-3 shape: (1516948, 16)
Hydraulic shape: (2205, 73)

MetroPT-3 failure events: 4
   failure_id          start_time            end_time failure_type  \
0           1 2020-04-18 00:00:00 2020-04-18 23:59:00     Air leak   
1           2 2020-05-29 23:30:00 2020-05-30 06:00:00     Air leak   
2           3 2020-06-05 10:00:00 2020-06-07 14:30:00     Air leak   
3           4 2020-07-15 14:30:00 2020-07-15 19:00:00     Air leak   

      severity                    report_note        duration  
0  High stress                                0 days 23:59:00  
1  High stress  Maintenance on 30Apr at 12:00 0 days 06:30:00  
2  High stress   Maintenance on 8Jun at 16:00 2 days 04:30:00  
3  High stress  Maintenance on 16Jul at 00:00 0 days 04:30:00  


In [2]:
#Quick check
print("\nMetro variables:", metro_vars)
print("Hydraulic comparisons:")
for label, feats in hydraulic_pairs.items():
    print(f"  {label}: {feats}")


Metro variables: ['TP2', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current']
Hydraulic comparisons:
  cooler_condition: ['TS1_mean', 'TS1_std']
  pump_leakage: ['EPS1_mean', 'PS1_mean']
  stable_flag: ['TS1_std', 'VS1_std', 'PS1_std', 'EPS1_std']


## MetroPT-3: pre-failure vs normal comparisons

The MetroPT-3 dataset contains known failure events but no direct row-level degradation labels. The main idea in this section is therefore to compare selected sensor variables between pre-failure windows and normal-operation windows.

Several candidate pre-failure window lengths may be tested in this scratchpad before one final definition is selected for the clean notebook.

### Failure-event definition

The MetroPT-3 sensor table is not labeled at the row level, but the dataset documentation provides four reported failure intervals. These are loaded as structured metadata and used to define pre-failure windows relative to the failure start time. The intervals themselves are treated as failure periods and should not be included in the normal-operation baseline.

In [3]:
# Candidate window extraction helpers

def extract_prefailure_windows(df, failure_df, hours):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < ft)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()


def extract_normal_windows(df, failure_df, hours, buffer_hours=48):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours + buffer_hours)
        end = ft - pd.Timedelta(hours=buffer_hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < end)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()

In [4]:
# Candidate window counts

for h in [1, 6, 12, 24]:
    prefail = extract_prefailure_windows(metro, metro_failures, h)
    normal = extract_normal_windows(metro, metro_failures, h, buffer_hours=48)

    print(f"{h}h window -> pre-failure rows: {len(prefail):,}, normal rows: {len(normal):,}")

1h window -> pre-failure rows: 1,387, normal rows: 1,025
6h window -> pre-failure rows: 8,327, normal rows: 7,773
12h window -> pre-failure rows: 16,654, normal rows: 16,030
24h window -> pre-failure rows: 28,668, normal rows: 26,829


In [5]:
# Inspecting one case

prefail_6h = extract_prefailure_windows(metro, metro_failures, 6)
normal_6h = extract_normal_windows(metro, metro_failures, 6, buffer_hours=48)

print("Pre-failure 6h sample:")
print(prefail_6h[['timestamp', 'failure_id', 'failure_start']].head())

print("\nNormal 6h sample:")
print(normal_6h[['timestamp', 'failure_id', 'failure_start']].head())

Pre-failure 6h sample:
            timestamp  failure_id failure_start
0 2020-04-17 18:00:10           1    2020-04-18
1 2020-04-17 18:00:22           1    2020-04-18
2 2020-04-17 18:00:34           1    2020-04-18
3 2020-04-17 18:00:46           1    2020-04-18
4 2020-04-17 18:00:58           1    2020-04-18

Normal 6h sample:
            timestamp  failure_id failure_start
0 2020-04-15 18:00:09           1    2020-04-18
1 2020-04-15 18:00:19           1    2020-04-18
2 2020-04-15 18:00:29           1    2020-04-18
3 2020-04-15 18:00:39           1    2020-04-18
4 2020-04-15 18:00:49           1    2020-04-18


In [6]:
# Catch/Print weird imbalances

print("Pre-failure rows per event (6h):")
print(prefail_6h['failure_id'].value_counts().sort_index())

print("\nNormal rows per event (6h):")
print(normal_6h['failure_id'].value_counts().sort_index())

Pre-failure rows per event (6h):
failure_id
1    1790
2    2179
3    2179
4    2179
Name: count, dtype: int64

Normal rows per event (6h):
failure_id
1    2179
2    1786
3    2179
4    1629
Name: count, dtype: int64


In [7]:
# Find timestamp gaps inside the 6h pre-failure and normal windows

def find_gaps_in_windows(df, window_label):
    """Find gaps > 30 seconds inside extracted windows, grouped by failure_id."""
    df = df.sort_values(['failure_id', 'timestamp']).reset_index(drop=True)
    df['gap'] = df.groupby('failure_id')['timestamp'].diff()
    gaps = df[df['gap'] > pd.Timedelta(seconds=30)]
    if len(gaps) == 0:
        print(f"{window_label}: no gaps > 30s inside any window")
    else:
        print(f"{window_label}: {len(gaps)} gap(s) found")
        print(gaps[['failure_id', 'timestamp', 'gap']])

find_gaps_in_windows(prefail_6h, "Pre-failure 6h")
print()
find_gaps_in_windows(normal_6h, "Normal 6h")

Pre-failure 6h: no gaps > 30s inside any window

Normal 6h: no gaps > 30s inside any window


In [8]:
print("Pre-failure 6h actual time spans per failure:")
for fid in prefail_6h['failure_id'].unique():
    sub = prefail_6h[prefail_6h['failure_id'] == fid]
    actual_start = sub['timestamp'].min()
    actual_end = sub['timestamp'].max()
    requested_start = sub['failure_start'].iloc[0] - pd.Timedelta(hours=6)
    requested_end = sub['failure_start'].iloc[0]
    print(f"  Failure {fid}:")
    print(f"    Requested: {requested_start} -> {requested_end}")
    print(f"    Actual:    {actual_start} -> {actual_end}")
    print(f"    Coverage:  {(actual_end - actual_start).total_seconds() / 3600:.2f}h of 6h")

print("\nNormal 6h actual time spans per failure:")
for fid in normal_6h['failure_id'].unique():
    sub = normal_6h[normal_6h['failure_id'] == fid]
    actual_start = sub['timestamp'].min()
    actual_end = sub['timestamp'].max()
    requested_end = sub['failure_start'].iloc[0] - pd.Timedelta(hours=48)
    requested_start = requested_end - pd.Timedelta(hours=6)
    print(f"  Failure {fid}:")
    print(f"    Requested: {requested_start} -> {requested_end}")
    print(f"    Actual:    {actual_start} -> {actual_end}")
    print(f"    Coverage:  {(actual_end - actual_start).total_seconds() / 3600:.2f}h of 6h")

Pre-failure 6h actual time spans per failure:
  Failure 1:
    Requested: 2020-04-17 18:00:00 -> 2020-04-18 00:00:00
    Actual:    2020-04-17 18:00:10 -> 2020-04-17 23:59:49
    Coverage:  5.99h of 6h
  Failure 2:
    Requested: 2020-05-29 17:30:00 -> 2020-05-29 23:30:00
    Actual:    2020-05-29 17:30:08 -> 2020-05-29 23:29:58
    Coverage:  6.00h of 6h
  Failure 3:
    Requested: 2020-06-05 04:00:00 -> 2020-06-05 10:00:00
    Actual:    2020-06-05 04:00:06 -> 2020-06-05 09:59:54
    Coverage:  6.00h of 6h
  Failure 4:
    Requested: 2020-07-15 08:30:00 -> 2020-07-15 14:30:00
    Actual:    2020-07-15 08:30:00 -> 2020-07-15 14:29:51
    Coverage:  6.00h of 6h

Normal 6h actual time spans per failure:
  Failure 1:
    Requested: 2020-04-15 18:00:00 -> 2020-04-16 00:00:00
    Actual:    2020-04-15 18:00:09 -> 2020-04-15 23:59:59
    Coverage:  6.00h of 6h
  Failure 2:
    Requested: 2020-05-27 17:30:00 -> 2020-05-27 23:30:00
    Actual:    2020-05-27 17:30:09 -> 2020-05-27 23:29:55
   

In [9]:
for fid in [1, 2, 4]:
    print(f"\nFailure {fid} pre-failure interval distribution:")
    sub = prefail_6h[prefail_6h['failure_id'] == fid].sort_values('timestamp')
    print(sub['timestamp'].diff().value_counts().head(10))

for fid in [2, 4]:
    print(f"\nFailure {fid} normal interval distribution:")
    sub = normal_6h[normal_6h['failure_id'] == fid].sort_values('timestamp')
    print(sub['timestamp'].diff().value_counts().head(10))


Failure 1 pre-failure interval distribution:
timestamp
0 days 00:00:12    1366
0 days 00:00:13     267
0 days 00:00:11     156
Name: count, dtype: int64

Failure 2 pre-failure interval distribution:
timestamp
0 days 00:00:10    1988
0 days 00:00:09     190
Name: count, dtype: int64

Failure 4 pre-failure interval distribution:
timestamp
0 days 00:00:10    1989
0 days 00:00:09     189
Name: count, dtype: int64

Failure 2 normal interval distribution:
timestamp
0 days 00:00:12    1313
0 days 00:00:13     319
0 days 00:00:11     153
Name: count, dtype: int64

Failure 4 normal interval distribution:
timestamp
0 days 00:00:10    1484
0 days 00:00:09     143
0 days 00:00:12       1
Name: count, dtype: int64


### Window-quality note

Initial diagnostics showed that lower row counts can arise either from slightly slower regular sampling (e.g. 11-13 s intervals) or from genuinely truncated windows. Because one normal 6-hour window was substantially incomplete, later statistical comparisons should be restricted to windows that satisfy a minimum coverage threshold.

In [10]:
def filter_windows_by_coverage(df, requested_hours, min_fraction=0.9):
    coverage = (
        df.groupby('failure_id')['timestamp']
        .agg(actual_start='min', actual_end='max')
        .reset_index()
    )

    coverage['coverage_hours'] = (
        coverage['actual_end'] - coverage['actual_start']
    ).dt.total_seconds() / 3600

    coverage['keep'] = coverage['coverage_hours'] >= requested_hours * min_fraction

    keep_ids = coverage.loc[coverage['keep'], 'failure_id']
    filtered = df[df['failure_id'].isin(keep_ids)].copy()

    return filtered, coverage

In [11]:
# Apply coverage filter independently to pre-failure and normal windows
prefail_6h_filtered, prefail_coverage = filter_windows_by_coverage(
    prefail_6h, requested_hours=6
)
normal_6h_filtered, normal_coverage = filter_windows_by_coverage(
    normal_6h, requested_hours=6
)

# Keep only failures that survived both filters (pairing rule)
kept_ids = set(prefail_coverage.loc[prefail_coverage['keep'], 'failure_id']) & \
           set(normal_coverage.loc[normal_coverage['keep'], 'failure_id'])

prefail_6h_clean = prefail_6h_filtered[prefail_6h_filtered['failure_id'].isin(kept_ids)].copy()
normal_6h_clean  = normal_6h_filtered[normal_6h_filtered['failure_id'].isin(kept_ids)].copy()

print("Pre-failure coverage:")
print(prefail_coverage)

print("\nNormal coverage:")
print(normal_coverage)

print(f"\nFailure events surviving the coverage rule: {sorted(kept_ids)}")
print(f"Pre-failure rows: {len(prefail_6h_clean):,}")
print(f"Normal rows:      {len(normal_6h_clean):,}")

Pre-failure coverage:
   failure_id        actual_start          actual_end  coverage_hours  keep
0           1 2020-04-17 18:00:10 2020-04-17 23:59:49        5.994167  True
1           2 2020-05-29 17:30:08 2020-05-29 23:29:58        5.997222  True
2           3 2020-06-05 04:00:06 2020-06-05 09:59:54        5.996667  True
3           4 2020-07-15 08:30:00 2020-07-15 14:29:51        5.997500  True

Normal coverage:
   failure_id        actual_start          actual_end  coverage_hours   keep
0           1 2020-04-15 18:00:09 2020-04-15 23:59:59        5.997222   True
1           2 2020-05-27 17:30:09 2020-05-27 23:29:55        5.996111   True
2           3 2020-06-03 04:00:03 2020-06-03 09:59:54        5.997500   True
3           4 2020-07-13 08:30:06 2020-07-13 12:59:05        4.483056  False

Failure events surviving the coverage rule: [1, 2, 3]
Pre-failure rows: 6,148
Normal rows:      6,144


### Observations

The coverage filter retains three of four failure events at the 6-hour window length. Failure 4 is excluded because its paired normal window covers only 4.48 hours of the requested 6 hours, falling below the 90% coverage threshold. The pre-failure window for Failure 4 is dropped together with its normal counterpart to preserve symmetric pairing.

After filtering, the pre-failure and normal samples are closely matched in size (6,148 vs 6,144 rows), which supports a cleaner comparison between the two groups. However, because the rows within each window are temporally dependent, the next step will summarize each failure window at the event level before formal statistical testing.

The same coverage rule will be reapplied at every window length in the sensitivity sweep, and the number of retained failure events will be reported alongside the test results.

### Coverage-filter result

After applying a minimum coverage rule, only failures 1, 2, and 3 remained usable for matched 6-hour pre-failure versus normal comparison. Failure 4 was excluded because the corresponding normal window was substantially truncated.

In [12]:
# Summarize each event-window

def summarize_windows(df, variables, group_label):
    summary = (
        df.groupby('failure_id')[variables]
        .median()
        .reset_index()
    )
    summary['group'] = group_label
    return summary

In [13]:
prefail_6h_summary = summarize_windows(prefail_6h_clean, metro_vars, 'prefail_6h')
normal_6h_summary = summarize_windows(normal_6h_clean, metro_vars, 'normal_6h')

metro_event_summary_6h = pd.concat(
    [prefail_6h_summary, normal_6h_summary],
    ignore_index=True
)

metro_event_summary_6h

,failure_id,TP2,H1,DV_pressure,Oil_temperature,Motor_current,group
0,1,-0.018,8.238,-0.024,49.450,0.0400,prefail_6h
1,2,-0.012,8.764,-0.020,63.975,0.0425,prefail_6h
2,3,-0.012,8.800,-0.022,63.475,0.0450,prefail_6h
3,1,-0.014,8.886,-0.024,57.725,0.0425,normal_6h
4,2,-0.010,8.158,-0.018,64.600,0.0400,normal_6h
5,3,-0.012,8.864,-0.022,63.775,0.0425,normal_6h


In [14]:
# Checking whether the per-event medians move in the same direction

for var in metro_vars:
    print(f"\n{var}")
    print(
        metro_event_summary_6h[['failure_id', 'group', var]]
        .sort_values(['failure_id', 'group'])
        .to_string(index=False)
    )


TP2
 failure_id      group    TP2
          1  normal_6h -0.014
          1 prefail_6h -0.018
          2  normal_6h -0.010
          2 prefail_6h -0.012
          3  normal_6h -0.012
          3 prefail_6h -0.012

H1
 failure_id      group    H1
          1  normal_6h 8.886
          1 prefail_6h 8.238
          2  normal_6h 8.158
          2 prefail_6h 8.764
          3  normal_6h 8.864
          3 prefail_6h 8.800

DV_pressure
 failure_id      group  DV_pressure
          1  normal_6h       -0.024
          1 prefail_6h       -0.024
          2  normal_6h       -0.018
          2 prefail_6h       -0.020
          3  normal_6h       -0.022
          3 prefail_6h       -0.022

Oil_temperature
 failure_id      group  Oil_temperature
          1  normal_6h           57.725
          1 prefail_6h           49.450
          2  normal_6h           64.600
          2 prefail_6h           63.975
          3  normal_6h           63.775
          3 prefail_6h           63.475

Motor_current
 

### Observations

After summarizing the retained 6-hour windows at the failure-event level, the clearest and most consistent pre-failure shift appears in `Oil_temperature`: all three retained failures show lower median oil temperature in the pre-failure window than in the matched normal window. `H1` also shows a consistent downward shift across the three events.

`TP2` shows a weaker pattern, with a pre-failure decrease in two of the three failures but no change in the third. By contrast, `DV_pressure` shows almost no systematic difference, and `Motor_current` changes in inconsistent directions across events.

At the 6-hour window level, the most promising MetroPT-3 candidates for formal comparison therefore appear to be `Oil_temperature` and `H1`, with `TP2` as a weaker secondary candidate.

## Tomorrow's work
- Final MetroPT window choice: **6h** (12h as sensitivity check)
- Final MetroPT kept variables: **Primary: `Oil_temperature`, `TP2`. Secondary: `H1`. Dropped: `Motor_current`, `DV_pressure`.**
- Final hydraulic kept tests: **done**
- Reliability section scope: ** done**
- Items ready to move into clean Notebook 03:

In [15]:
# Window-length sweep: extract, filter by coverage, compute per-event medians

WINDOW_LENGTHS_HOURS = [1, 6, 12, 24]
COVERAGE_THRESHOLD = 0.9

sweep_records = []
coverage_records = []
sweep_kept_ids = {}  # window_length -> kept failure_ids

for h in WINDOW_LENGTHS_HOURS:
    # Extract windows
    prefail = extract_prefailure_windows(metro, metro_failures, h)
    normal  = extract_normal_windows(metro, metro_failures, h, buffer_hours=48)

    # Apply coverage filter independently
    prefail_filt, prefail_cov = filter_windows_by_coverage(
        prefail, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )
    normal_filt, normal_cov = filter_windows_by_coverage(
        normal, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )

    prefail_cov['window_h'] = h
    prefail_cov['group']    = 'prefail'
    normal_cov['window_h']  = h
    normal_cov['group']     = 'normal'
    coverage_records.append(prefail_cov)
    coverage_records.append(normal_cov)

    # Enforce matched failures
    kept_ids = sorted(
        set(prefail_cov.loc[prefail_cov['keep'], 'failure_id']) &
        set(normal_cov.loc[normal_cov['keep'], 'failure_id'])
    )
    sweep_kept_ids[h] = kept_ids

    prefail_clean = prefail_filt[prefail_filt['failure_id'].isin(kept_ids)].copy()
    normal_clean  = normal_filt[normal_filt['failure_id'].isin(kept_ids)].copy()

    # Per-event medians
    for fid in kept_ids:
        for var in metro_vars:
            pre_median = prefail_clean.loc[
                prefail_clean['failure_id'] == fid, var
            ].median()
            norm_median = normal_clean.loc[
                normal_clean['failure_id'] == fid, var
            ].median()

            sweep_records.append({
                'window_h':             h,
                'n_kept_failures':      len(kept_ids),
                'failure_id':           fid,
                'variable':             var,
                'prefail_median':       pre_median,
                'normal_median':        norm_median,
                'prefail_minus_normal': pre_median - norm_median,
            })

sweep_df = pd.DataFrame(sweep_records)
coverage_df = pd.concat(coverage_records, ignore_index=True)

print("Failures retained per window length:")
for h in WINDOW_LENGTHS_HOURS:
    print(f"  {h:>2}h: {sweep_kept_ids[h]}")

print(f"\nTotal sweep records: {len(sweep_df)}")
print("\nSweep dataframe head:")
print(sweep_df.head(15))

Failures retained per window length:
   1h: [1, 2, 3]
   6h: [1, 2, 3]
  12h: [1, 2, 3]
  24h: [1, 2, 3]

Total sweep records: 60

Sweep dataframe head:
    window_h  n_kept_failures  failure_id         variable  prefail_median  \
0          1                3           1              TP2          -0.018   
1          1                3           1               H1           8.238   
2          1                3           1      DV_pressure          -0.024   
3          1                3           1  Oil_temperature          49.450   
4          1                3           1    Motor_current           0.040   
5          1                3           2              TP2          -0.012   
6          1                3           2               H1           8.450   
7          1                3           2      DV_pressure          -0.020   
8          1                3           2  Oil_temperature          63.650   
9          1                3           2    Motor_current         

**For each variable, at which window length is the direction consistency strongest, and is the median magnitude meaningful?**

In [16]:
consistency_summary = (
    sweep_df.groupby(['window_h', 'variable'])['prefail_minus_normal']
    .agg(
        median_diff='median',
        n_negative=lambda x: (x < 0).sum(),
        n_positive=lambda x: (x > 0).sum(),
        n_zero=lambda x: (x == 0).sum()
    )
    .reset_index()
)

consistency_summary


,window_h,variable,median_diff,n_negative,n_positive,n_zero
0,1,DV_pressure,0.0000,1,0,2
1,1,H1,-0.4580,2,1,0
2,1,Motor_current,0.0025,1,2,0
3,1,Oil_temperature,-0.9500,2,1,0
4,1,TP2,-0.0020,2,0,1
5,6,DV_pressure,0.0000,1,0,2
6,6,H1,-0.0640,2,1,0
7,6,Motor_current,0.0025,1,2,0
8,6,Oil_temperature,-0.6250,3,0,0
9,6,TP2,-0.0020,2,0,1


In [17]:
# Sorting
consistency_summary.sort_values(['variable', 'window_h'])

,window_h,variable,median_diff,n_negative,n_positive,n_zero
0,1,DV_pressure,0.0000,1,0,2
5,6,DV_pressure,0.0000,1,0,2
10,12,DV_pressure,0.0000,1,0,2
15,24,DV_pressure,0.0000,0,0,3
1,1,H1,-0.4580,2,1,0
6,6,H1,-0.0640,2,1,0
11,12,H1,-0.0120,2,1,0
16,24,H1,0.0600,1,2,0
2,1,Motor_current,0.0025,1,2,0
7,6,Motor_current,0.0025,1,2,0


In [18]:
# Consistency count

# Direction summary string per (variable, window_h)
consistency_summary['direction_summary'] = (
    consistency_summary['n_negative'].astype(str) + 'n / ' +
    consistency_summary['n_positive'].astype(str) + 'p / ' +
    consistency_summary['n_zero'].astype(str) + 'z'
)

direction_pivot = consistency_summary.pivot(
    index='variable',
    columns='window_h',
    values='direction_summary'
)
direction_pivot

window_h,1,6,12,24
variable,,,,
DV_pressure,1n / 0p / 2z,1n / 0p / 2z,1n / 0p / 2z,0n / 0p / 3z
H1,2n / 1p / 0z,2n / 1p / 0z,2n / 1p / 0z,1n / 2p / 0z
Motor_current,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z
Oil_temperature,2n / 1p / 0z,3n / 0p / 0z,2n / 1p / 0z,1n / 2p / 0z
TP2,2n / 0p / 1z,2n / 0p / 1z,1n / 0p / 2z,1n / 0p / 2z


### Observations

The window-length sweep suggests that the 6-hour pre-failure window provides the clearest and most interpretable MetroPT-3 signal. `Oil_temperature` shows the strongest consistency at this window length, with all three retained failures exhibiting lower pre-failure medians than their matched normal windows. `H1` shows a weaker but still partly consistent downward shift across the shorter windows.

By contrast, `Motor_current` shows mixed direction across events at every tested window length, indicating that its variation reflects operating-regime changes rather than a pre-failure signature. `DV_pressure` shows almost no systematic change between pre-failure and normal windows at any window length and shows no observable pre-failure signal in this analysis. `TP2` shows a partial signal (two of three failures with lower pre-failure medians, the third flat), placing it as a weaker but coherent secondary candidate. `H1` shows partial consistency at the shorter windows but with a direction flip in one failure, weakening its interpretation.

Based on this sweep, the 6-hour window will be treated as the main MetroPT-3 comparison window, while 12 hours may be retained as a sensitivity check if needed.

### Event-level Mann-Whitney U at the 6-hour window

The formal hypothesis tests for MetroPT-3 are conducted at the event-summary level, using the per-failure median as the unit of comparison rather than individual sensor rows. This protects against the within-window autocorrelation that would inflate row-level significance, but reduces sample size to n=3 vs n=3 after the coverage filter, which limits the statistical power available.

Three variables are tested: `Oil_temperature`, `TP2`, and `H1`. `Motor_current` and `DV_pressure` were excluded based on the window-length sweep findings reported above. Bonferroni correction across $k=3$ tests sets the corrected significance threshold at $\alpha = 0.05/3 \approx 0.0167$.

Per the planned interpretation rule, results are evaluated through the combination of significance, direction, magnitude, and physical plausibility, with significance treated as one criterion among several rather than as the decisive measure.

In [19]:
def mannwhitney_table(df_a, df_b, variables, group_a='A', group_b='B'):
    """Mann-Whitney U for each variable, with Bonferroni-corrected
    p-values, rank-biserial effect size, sample sizes, and direction."""
    results = []
    for var in variables:
        x = df_a[var].dropna()
        y = df_b[var].dropna()

        stat, p = mannwhitneyu(x, y, alternative='two-sided')

        n1, n2 = len(x), len(y)
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        diff = x.median() - y.median()
        if diff > 0:
            direction = f'{group_a} > {group_b}'
        elif diff < 0:
            direction = f'{group_b} > {group_a}'
        else:
            direction = 'equal'

        results.append({
            'variable':           var,
            f'n_{group_a}':       n1,
            f'n_{group_b}':       n2,
            f'{group_a}_median':  x.median(),
            f'{group_b}_median':  y.median(),
            'direction':          direction,
            'u_stat':             stat,
            'p_value':            p,
            'rank_biserial':      rank_biserial,
        })

    out = pd.DataFrame(results)
    out['p_bonf'] = multipletests(out['p_value'], method='bonferroni')[1]
    return out.sort_values('p_value')

In [20]:
metro_vars_final_6h = ['Oil_temperature', 'TP2', 'H1']

In [21]:
metro_event_results_6h = mannwhitney_table(
    prefail_6h_summary,
    normal_6h_summary,
    metro_vars_final_6h,
    group_a='prefail',
    group_b='normal'
)

metro_event_results_6h

,variable,n_prefail,n_normal,prefail_median,normal_median,direction,u_stat,p_value,rank_biserial,p_bonf
1,TP2,3,3,-0.012,-0.012,equal,3.0,0.642835,0.333333,1.0
0,Oil_temperature,3,3,63.475,63.775,normal > prefail,3.0,0.700000,0.333333,1.0
2,H1,3,3,8.764,8.864,normal > prefail,3.0,0.700000,0.333333,1.0


### Observations

The event-level Mann-Whitney U results for the retained 6-hour MetroPT-3 windows do not reach statistical significance after Bonferroni correction. This is expected, given that only three matched failure events remain available after the coverage filter, which makes formal inference very low-powered.

Even so, the direction of the event-level medians remains informative. `Oil_temperature` and `H1` both show lower pre-failure medians than their matched normal windows, consistent with the earlier descriptive inspection. By contrast, `TP2` does not show a clear difference at the grouped median level, even though the per-event view showed lower pre-failure medians in two of the three retained failures. This illustrates that the median-of-three event summary is a stricter statistic than simple direction counting and may hide partial shifts that are not consistent across most events.

At this stage, the MetroPT-3 evidence is therefore better interpreted as directional and exploratory-supportive rather than as strong standalone statistical confirmation. In the final notebook, the 6-hour window can still be retained as the main MetroPT comparison, but the limited number of matched failure events must be stated explicitly when discussing the test results.

## Hydraulic system: condition-based comparisons

The hydraulic dataset is cycle-based and includes explicit condition labels for cooler condition, valve condition, pump leakage, accumulator pressure, and overall system stability. Unlike MetroPT-3, no temporal window construction is needed: each row of the cycle-aggregated feature table corresponds to a single labeled cycle, and groups are defined directly by label values.

The exploratory analysis in Notebook 02 identified the strongest feature-condition relationships as `TS1_mean` and `TS1_std` for `cooler_condition`, `EPS1_mean` for `pump_leakage`, and standard-deviation features for `stable_flag`. `valve_condition` and `accumulator_pressure` showed weaker EDA separation and are not formally tested, but are reported descriptively as contrastive findings.

Multi-level labels are handled according to the test that best matches their structure. Binary contrasts use the Mann-Whitney U test; multi-level contrasts use the Kruskal-Wallis omnibus test. After that pairwise Mann-Whitney comparisons are run only when the omnibus result is significant after correction.

### Label structure

Before defining the formal tests, the label-level distributions are inspected to confirm the number of levels and group balance for each condition variable.

In [22]:
# Inspect hydraulic label structure
hydraulic_label_cols = ['cooler_condition', 'valve_condition',
                        'pump_leakage', 'accumulator_pressure', 'stable_flag']

for col in hydraulic_label_cols:
    counts = hydraulics[col].value_counts().sort_index()
    print(f"{col}:")
    print(counts.to_string())
    print()

cooler_condition:
cooler_condition
3      732
20     732
100    741

valve_condition:
valve_condition
73      360
80      360
90      360
100    1125

pump_leakage:
pump_leakage
0    1221
1     492
2     492

accumulator_pressure:
accumulator_pressure
90     808
100    399
115    399
130    599

stable_flag:
stable_flag
0    1449
1     756



### Observations

Each label has a distinct structure. `cooler_condition` is a three-level variable with near-perfect group balance (732, 732, 741 cycles), well-suited to a Kruskal-Wallis omnibus test with possible follow-up pairwise comparisons. `pump_leakage` is a three-level variable dominated by the healthy class (1221 cycles vs 492 each for weak and severe leakage); the cleanest test is an endpoint contrast between full health (0) and severe leakage (2). `stable_flag` is already binary (1449 stable, 756 unstable) and tests directly with Mann-Whitney U.

`valve_condition` and `accumulator_pressure` show four-level imbalanced distributions and were already flagged as weaker EDA candidates in Notebook 02. Both are excluded from formal testing and reported descriptively only.

The total formal-test count is five: two features for `cooler_condition`, one for `pump_leakage`, and two for `stable_flag`. The Bonferroni-corrected significance threshold for primary tests is therefore $\alpha = 0.05 / 5 = 0.010$. Optional features within the same physical concept are run as exploratory checks and are not included in the Bonferroni count.

### Cooler condition: formal testing

The `cooler_condition` label has three levels (3, 20, 100) representing close-to-total cooler failure, reduced efficiency, and full efficiency. The Kruskal-Wallis omnibus test is applied to the two strongest features identified in Notebook 02: `TS1_mean` and `TS1_std`. If the omnibus result is significant after correction, follow-up pairwise Mann-Whitney comparisons are run between each pair of cooler-condition levels.

In [23]:
# Kruskal-Wallis on TS1_mean and TS1_std by cooler_condition
# Global Bonferroni correction across k = 5 primary hydraulic tests

GLOBAL_K = 5
GLOBAL_ALPHA = 0.05 / GLOBAL_K  # 0.010

cooler_features = ['TS1_mean', 'TS1_std']
cooler_levels = sorted(hydraulics['cooler_condition'].unique())

cooler_kw_results = []

for var in cooler_features:
    groups = [
        hydraulics.loc[hydraulics['cooler_condition'] == lvl, var].dropna()
        for lvl in cooler_levels
    ]
    stat, p = kruskal(*groups)

    cooler_kw_results.append({
        'variable': var,
        'levels': cooler_levels,
        'group_sizes': [len(g) for g in groups],
        'group_medians': [g.median() for g in groups],
        'H_statistic': stat,
        'p_value': p,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

cooler_kw_df = pd.DataFrame(cooler_kw_results)
# Clean display formatting
cooler_kw_df['group_medians'] = cooler_kw_df['group_medians'].apply(
    lambda vals: [round(v, 4) for v in vals]
)
cooler_kw_df['H_statistic'] = cooler_kw_df['H_statistic'].round(4)
cooler_kw_df['p_value'] = cooler_kw_df['p_value'].apply(lambda p: f"{p:.3e}")

cooler_kw_df

,variable,levels,group_sizes,group_medians,H_statistic,p_value,alpha_bonf,significant
0,TS1_mean,"[3, 20, 100]","[732, 732, 741]","[55.6925, 44.9063, 35.8881]",1915.5335,0.000e+00,0.01,True
1,TS1_std,"[3, 20, 100]","[732, 732, 741]","[0.2279, 0.1961, 0.1507]",1703.3084,0.000e+00,0.01,True


**Observations**

The Kruskal-Wallis omnibus test produces highly significant results for both `TS1_mean` and `TS1_std` (`H = 1915.5` and `H = 1703.3`, respectively, with both `p` values effectively zero). Both results are far below the Bonferroni-corrected threshold of \(\alpha = 0.010\), providing strong support for the hydraulic part of Hypothesis 1 in the case of `cooler_condition`.

The group medians show clear monotonic separation across cooler-efficiency levels. For `TS1_mean`, the median temperature decreases from 55.7°C (3% efficiency) through 44.9°C (20% efficiency) to 35.9°C (full efficiency), consistent with the physical expectation that a degraded cooler fails to dissipate heat effectively. For `TS1_std`, the within-cycle variability also decreases monotonically, from approximately 0.228 to 0.196 and then to 0.151, indicating that poorer cooler condition is associated not only with higher temperature but also with less stable thermal behavior during each cycle.

These findings are the strongest formal hypothesis-test results obtained so far. They confirm the cooler-condition pattern identified descriptively in Notebook 02 and show that engineered per-cycle features can separate degradation states very clearly for at least one hydraulic component.

### Pairwise comparisons

Because the omnibus Kruskal-Wallis result is significant for both cooler-related features, pairwise Mann-Whitney comparisons are then run to confirm whether the separation holds across each pair of cooler-condition levels.

In [24]:
# Pairwise Mann-Whitney for cooler_condition
# Pairs: 3 vs 20, 3 vs 100, 20 vs 100
# Bonferroni correction within the pairwise comparison family:
# 2 features × 3 pairs = 6 tests

cooler_pairs = [(3, 20), (3, 100), (20, 100)]
cooler_features = ['TS1_mean', 'TS1_std']
PAIRWISE_ALPHA = 0.05 / (len(cooler_pairs) * len(cooler_features))  # 0.05 / 6

cooler_pairwise_results = []

for var in cooler_features:
    for a, b in cooler_pairs:
        group_a = hydraulics.loc[hydraulics['cooler_condition'] == a, var].dropna()
        group_b = hydraulics.loc[hydraulics['cooler_condition'] == b, var].dropna()

        stat, p = mannwhitneyu(group_a, group_b, alternative='two-sided')

        n1, n2 = len(group_a), len(group_b)
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        med_a = group_a.median()
        med_b = group_b.median()
        if med_a > med_b:
            direction = f'{a} > {b}'
        elif med_a < med_b:
            direction = f'{b} > {a}'
        else:
            direction = 'equal'

        cooler_pairwise_results.append({
            'variable':      var,
            'comparison':    f'{a} vs {b}',
            'n_a':           n1,
            'n_b':           n2,
            'median_a':      med_a,
            'median_b':      med_b,
            'direction':     direction,
            'u_stat':        stat,
            'p_value':       p,
            'rank_biserial': rank_biserial,
            'alpha_bonf':    PAIRWISE_ALPHA,
            'significant':   p < PAIRWISE_ALPHA,
        })

cooler_pairwise_df = pd.DataFrame(cooler_pairwise_results)

# Clean display formatting
for col in ['median_a', 'median_b', 'u_stat', 'rank_biserial']:
    cooler_pairwise_df[col] = cooler_pairwise_df[col].round(4)
cooler_pairwise_df['p_value'] = cooler_pairwise_df['p_value'].apply(lambda p: f"{p:.3e}")

cooler_pairwise_df

,variable,comparison,n_a,n_b,median_a,median_b,direction,u_stat,p_value,rank_biserial,alpha_bonf,significant
0,TS1_mean,3 vs 20,732,732,55.6925,44.9063,3 > 20,524929.0,1.318e-221,-0.9593,0.008333,True
1,TS1_mean,3 vs 100,732,741,55.6925,35.8881,3 > 100,541875.0,4.060e-241,-0.9980,0.008333,True
2,TS1_mean,20 vs 100,732,741,44.9063,35.8881,20 > 100,542100.0,1.625e-241,-0.9988,0.008333,True
3,TS1_std,3 vs 20,732,732,0.2279,0.1961,3 > 20,475016.0,1.295e-144,-0.7730,0.008333,True
4,TS1_std,3 vs 100,732,741,0.2279,0.1507,3 > 100,540836.0,2.753e-239,-0.9942,0.008333,True
5,TS1_std,20 vs 100,732,741,0.1961,0.1507,20 > 100,529664.0,4.859e-220,-0.9530,0.008333,True


In [25]:
# Compact view for inline reading
cooler_pairwise_df[
    ['variable', 'comparison', 'median_a', 'median_b', 'direction', 'p_value', 'significant']
]

,variable,comparison,median_a,median_b,direction,p_value,significant
0,TS1_mean,3 vs 20,55.6925,44.9063,3 > 20,1.318e-221,True
1,TS1_mean,3 vs 100,55.6925,35.8881,3 > 100,4.060e-241,True
2,TS1_mean,20 vs 100,44.9063,35.8881,20 > 100,1.625e-241,True
3,TS1_std,3 vs 20,0.2279,0.1961,3 > 20,1.295e-144,True
4,TS1_std,3 vs 100,0.2279,0.1507,3 > 100,2.753e-239,True
5,TS1_std,20 vs 100,0.1961,0.1507,20 > 100,4.859e-220,True


**Observations**

All six pairwise Mann-Whitney comparisons remain significant under the Bonferroni-corrected threshold \(\alpha = 0.0083\), with p-values ranging from \(10^{-144}\) to \(10^{-241}\). The direction of every comparison is consistent with the monotonic gradient observed in the omnibus medians: more degraded cooler states show higher `TS1_mean` and `TS1_std` values across every pair of levels.

Rank-biserial effect sizes are very large in absolute magnitude, exceeding 0.95 for five of the six comparisons. The exception is `TS1_std` between cooler states 3% and 20%, where the rank-biserial value of about 0.77 still indicates a large effect, but a meaningfully weaker one than in the other contrasts. This suggests that within-cycle thermal variability separates the two most degraded cooler states less cleanly than mean temperature does, while both features clearly separate degraded states from full efficiency.

These pairwise results confirm that the cooler-condition signal is graded across all three degradation levels, rather than being driven only by an extreme best-versus-worst contrast. Both `TS1_mean` and `TS1_std` therefore qualify as strong discriminators of cooler condition, with `TS1_mean` appearing slightly preferable when distinguishing between the two non-healthy degradation states.

### Pump leakage

The `pump_leakage` label has three levels: `0` (no leakage), `1` (weak leakage), and `2` (severe leakage). Notebook 02 identified `EPS1_mean` as the strongest engineered feature for this condition. Because the healthy class dominates the cycle distribution (1221 cycles versus 492 each for weak and severe leakage), the cleanest formal contrast is the endpoint comparison between full health (`0`) and severe leakage (`2`). The weak leakage state (`1`) is omitted from the formal test but may still be referenced descriptively where relevant.

In [26]:
# Mann-Whitney U for pump_leakage: endpoint contrast 0 vs 2
# Bonferroni correction uses the global k = 5 family

pump_features = ['EPS1_mean']

pump_results = []

for var in pump_features:
    healthy = hydraulics.loc[hydraulics['pump_leakage'] == 0, var].dropna()
    severe = hydraulics.loc[hydraulics['pump_leakage'] == 2, var].dropna()

    stat, p = mannwhitneyu(healthy, severe, alternative='two-sided')

    n1, n2 = len(healthy), len(severe)
    rank_biserial = 1 - (2 * stat) / (n1 * n2)

    med_h = healthy.median()
    med_s = severe.median()

    if med_h > med_s:
        direction = 'healthy > severe'
    elif med_h < med_s:
        direction = 'severe > healthy'
    else:
        direction = 'equal'

    pump_results.append({
        'variable': var,
        'comparison': '0 (healthy) vs 2 (severe leakage)',
        'n_healthy': n1,
        'n_severe': n2,
        'median_healthy': med_h,
        'median_severe': med_s,
        'direction': direction,
        'u_stat': stat,
        'p_value': p,
        'rank_biserial': rank_biserial,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

pump_df = pd.DataFrame(pump_results)

# Clean display formatting
pump_df['median_healthy'] = pump_df['median_healthy'].round(4)
pump_df['median_severe'] = pump_df['median_severe'].round(4)
pump_df['u_stat'] = pump_df['u_stat'].round(4)
pump_df['rank_biserial'] = pump_df['rank_biserial'].round(4)
pump_df['p_value'] = pump_df['p_value'].apply(lambda p: f"{p:.3e}")

pump_df

,variable,comparison,n_healthy,n_severe,median_healthy,median_severe,direction,u_stat,p_value,rank_biserial,alpha_bonf,significant
0,EPS1_mean,0 (healthy) vs 2 (severe leakage),1221,492,2455.9669,2557.5761,severe > healthy,115092.0,5.366e-89,0.6168,0.01,True


**Observations**

The Mann-Whitney U test produces a highly significant result for `EPS1_mean` between the healthy and severely-leaking pump states ($U = 115092$, $p \approx 5 \times 10^{-89}$, far below the corrected threshold $\alpha = 0.010$). This provides strong support for the hydraulic part of Hypothesis 1 in the case of `pump_leakage` at the endpoint contrast.

The direction of the effect is physically interpretable. Median motor-power-related load is approximately 2557.6 in the severe-leakage group and 2456.0 in the healthy group, indicating that a leaking pump is associated with higher sustained motor load. This is consistent with the expectation that degraded pump performance increases the power demand needed to maintain operation.

The rank-biserial effect size of 0.62 lies in the large range, but is meaningfully smaller than the effect sizes obtained for `cooler_condition` (above 0.95 for most pairwise comparisons). This indicates that while the pump-leakage signal is statistically robust, the separation between healthy and severely leaking pumps is less clean than the separation observed for cooler-condition states. The weak-leakage state (`1`) is not formally tested here, but would be expected to lie between these two endpoints.

### Stable vs unstable cycles

The `stable_flag` label is a binary indicator distinguishing cycles judged stable (`0`) from unstable (`1`) by the original dataset annotators, where the unstable category corresponds to cycles in which static operating conditions had not been fully reached. Notebook 02 identified cycle-level standard-deviation features as the strongest descriptive separators of this label. Two features are tested formally: `TS1_std` and `VS1_std`, representing temperature variability and vibration variability, respectively.

In [27]:
# Mann-Whitney U for stable_flag: binary contrast 0 (stable) vs 1 (unstable)
# Bonferroni correction uses the global k = 5 family

stable_features = ['TS1_std', 'VS1_std']

stable_results = []

for var in stable_features:
    stable = hydraulics.loc[hydraulics['stable_flag'] == 0, var].dropna()
    unstable = hydraulics.loc[hydraulics['stable_flag'] == 1, var].dropna()

    stat, p = mannwhitneyu(stable, unstable, alternative='two-sided')

    n1, n2 = len(stable), len(unstable)
    rank_biserial = 1 - (2 * stat) / (n1 * n2)

    med_s = stable.median()
    med_u = unstable.median()

    if med_s > med_u:
        direction = 'stable > unstable'
    elif med_s < med_u:
        direction = 'unstable > stable'
    else:
        direction = 'equal'

    stable_results.append({
        'variable': var,
        'comparison': '0 (stable) vs 1 (unstable)',
        'n_stable': n1,
        'n_unstable': n2,
        'median_stable': med_s,
        'median_unstable': med_u,
        'direction': direction,
        'u_stat': stat,
        'p_value': p,
        'rank_biserial': rank_biserial,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

stable_df = pd.DataFrame(stable_results)

stable_df_display = stable_df.copy()


stable_df_display['median_stable'] = stable_df_display['median_stable'].round(4)
stable_df_display['median_unstable'] = stable_df_display['median_unstable'].round(4)
stable_df_display['u_stat'] = stable_df_display['u_stat'].round(4)
stable_df_display['rank_biserial'] = stable_df_display['rank_biserial'].round(4)
stable_df_display['p_value'] = stable_df_display['p_value'].apply(lambda p: f"{p:.3e}")

stable_df_display[
    ['variable', 'comparison', 'median_stable', 'median_unstable',
     'direction', 'p_value', 'rank_biserial', 'significant']
]

,variable,comparison,median_stable,median_unstable,direction,p_value,rank_biserial,significant
0,TS1_std,0 (stable) vs 1 (unstable),0.1922,0.2002,unstable > stable,5.584e-13,0.1868,True
1,VS1_std,0 (stable) vs 1 (unstable),0.0316,0.0367,unstable > stable,1.362e-04,0.0988,True


**Observations**

The Mann-Whitney U test produces significant results for both `TS1_std` and `VS1_std` between stable and unstable cycles (p-values $5.6 \times 10^{-13}$ and $1.4 \times 10^{-4}$, both below $\alpha = 0.010$). These results support the hydraulic part of Hypothesis 1 in the case of `stable_flag`.

The direction of effect is consistent for both features: unstable cycles show higher within-cycle variability than stable cycles, in line with the physical interpretation that unstable cycles correspond to transient periods in which static operating conditions have not been fully reached.

The rank-biserial effect sizes are notably small, however: 0.19 for `TS1_std` and 0.10 for `VS1_std`. Compared with the very large effects observed for `cooler_condition` (above 0.95) and the moderate-to-large effect for `pump_leakage` (0.62), the `stable_flag` distinction produces only modest separation. The result is statistically robust because of the large sample sizes, but the operational separation between stable and unstable cycles is weak: most unstable cycles still overlap heavily with the stable-cycle distribution. This is consistent with the descriptive Notebook 02 finding that temperature and vibration variability are only weak-to-moderate indicators of cycle stability.

## Hydraulic system: summary of formal results

Three condition labels were tested formally in the hydraulic dataset, with results that vary substantially in strength.

`cooler_condition` produced the strongest formal evidence in the entire project. Both `TS1_mean` and `TS1_std` separated the three cooler levels overwhelmingly (Kruskal-Wallis $p < 10^{-300}$), with rank-biserial effect sizes above 0.95 in five of the six pairwise comparisons. The signal is graded across all three degradation states, and the separation is physically interpretable as the thermal consequence of reduced cooler efficiency.

`pump_leakage` produced a robust but weaker result. `EPS1_mean` distinguished healthy from severe leakage with $p \approx 5 \times 10^{-89}$ and a large effect size of 0.62. The direction (severe > healthy in motor power) is consistent with the physical expectation that a degraded pump prompts higher motor load. The middle leakage state was not formally tested.

`stable_flag` produced statistically significant results that are operationally weak. Both `TS1_std` and `VS1_std` distinguished stable from unstable cycles ($p < 10^{-3}$), with the predicted direction (unstable > stable in within-cycle variability), but the rank-biserial effect sizes (0.19 and 0.10) indicate that individual unstable cycles overlap heavily with the stable distribution. The result confirms the descriptive Notebook 02 finding that cycle stability is a weak-to-moderate distinguisher.

Taken together, the three labels span a clear gradient of signal strength: the cooler-condition signal is dominant, the pump-leakage signal is robust, and the stable-flag signal is detectable but weak. All three results support Hypothesis 1 for the hydraulic dataset, though the operational meaningfulness of that support varies considerably across labels.

## Reliability metrics

Two reliability-oriented measures are calculated as supporting context for Hypothesis 3: Mean Time To Repair (MTTR) and Mean Time Between Failures (MTBF). For MetroPT-3, both metrics are computed directly from the four documented failure events. For the hydraulic dataset, neither metric translates cleanly because the data was produced under controlled laboratory conditions rather than as a continuous operational record; what can be reported instead is the proportion of cycles in each labeled degradation state. The asymmetry between the two datasets is acknowledged explicitly in the observation block.

### MetroPT-3 reliability

MetroPT-3 includes four documented air-leak failure events with start and end timestamps. These allow rough estimation of repair duration and time between failures. Because the number of events is small, the resulting metrics should be interpreted as descriptive context rather than as stable engineering reliability estimates.

In [28]:
metro_failures_sorted = metro_failures.sort_values('start_time').reset_index(drop=True)

# MTTR: mean of failure durations
metro_mttr = metro_failures_sorted['duration'].mean()

# MTBF: gaps between consecutive failures (end of failure i -> start of failure i+1)
gaps = (
    metro_failures_sorted['start_time'].iloc[1:].values -
    metro_failures_sorted['end_time'].iloc[:-1].values
)
metro_mtbf = pd.Series(pd.to_timedelta(gaps)).mean()

# Per-failure detail for the report
metro_failures_sorted['gap_to_next'] = pd.Series(
    [pd.NaT] * len(metro_failures_sorted),
    dtype='timedelta64[ns]'
)
metro_failures_sorted.loc[:len(metro_failures_sorted) - 2, 'gap_to_next'] = pd.to_timedelta(gaps)

print(f"MetroPT-3 reliability metrics (n = {len(metro_failures_sorted)} failures)\n")
print(f"  MTTR (Mean Time To Repair):    {metro_mttr}")
print(f"  MTBF (Mean Time Between Failures): {metro_mtbf}\n")

print("Per-failure detail:")
display_cols = ['failure_id', 'start_time', 'end_time', 'duration', 'gap_to_next']
print(metro_failures_sorted[display_cols].to_string(index=False))

MetroPT-3 reliability metrics (n = 4 failures)

  MTTR (Mean Time To Repair):    0 days 21:52:15
  MTBF (Mean Time Between Failures): 28 days 09:10:20

Per-failure detail:
 failure_id          start_time            end_time        duration      gap_to_next
          1 2020-04-18 00:00:00 2020-04-18 23:59:00 0 days 23:59:00 40 days 23:31:00
          2 2020-05-29 23:30:00 2020-05-30 06:00:00 0 days 06:30:00  6 days 04:00:00
          3 2020-06-05 10:00:00 2020-06-07 14:30:00 2 days 04:30:00 38 days 00:00:00
          4 2020-07-15 14:30:00 2020-07-15 19:00:00 0 days 04:30:00              NaT


**Observations**

Using the documented MetroPT-3 failure intervals, the average repair duration is approximately 21 hours and 25 minutes, while the average time between consecutive failures is about 28 days and 9 hours. These values should be interpreted descriptively because they are based on only four reported incidents, but they still provide useful operational context.

The MTTR estimate shows that the documented air-leak failures were not instantaneous anomalies; they corresponded to extended interruption or repair periods. The MTBF estimate, in turn, indicates that failures were relatively infrequent across the seven-month monitoring period, occurring roughly every four weeks on average.

Within this project, these reliability metrics do not serve as primary evidence on their own. Instead, they support the broader interpretation that the monitored compressor experienced intermittent but operationally meaningful failure episodes, which is consistent with the sensor-based analysis of pre-failure behavior.

### Hydraulic context

The hydraulic dataset is cycle-based rather than a continuous operational record with timestamped failures and repairs. For that reason, standard MTBF and MTTR definitions do not apply directly. Instead, this section reports the proportion of cycles observed in each relevant degradation or stability state. These values provide condition context, but they should not be interpreted as time-based reliability metrics comparable to those calculated for MetroPT-3.

In [29]:
cooler_state_share = (
    hydraulics['cooler_condition']
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

pump_state_share = (
    hydraulics['pump_leakage']
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

stable_state_share = (
    hydraulics['stable_flag']
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("Hydraulic dataset condition state shares (proportion of cycles)\n")

print("Cooler condition:")
print(cooler_state_share.to_string())

print("\nPump leakage:")
print(pump_state_share.to_string())

print("\nStable flag:")
print(stable_state_share.to_string())

Hydraulic dataset condition state shares (proportion of cycles)

Cooler condition:
cooler_condition
3      0.3320
20     0.3320
100    0.3361

Pump leakage:
pump_leakage
0    0.5537
1    0.2231
2    0.2231

Stable flag:
stable_flag
0    0.6571
1    0.3429


**Observations**

The hydraulic dataset provides substantial representation of both healthy and degraded operating states, but these proportions should be interpreted as condition context rather than as time-based reliability metrics.

`cooler_condition` is almost perfectly balanced across its three levels, which supports fair comparison of thermal degradation states. `pump_leakage` is more concentrated in the healthy class, but both weak and severe leakage remain well represented. `stable_flag` shows that about one-third of all cycles are unstable, confirming that the dataset contains a meaningful share of non-steady operating behavior.

These proportions help explain why the hydraulic dataset is well suited to condition-discrimination testing, but they are not directly comparable to MTBF or MTTR because the data does not describe a continuous operational timeline.

## Cross-dataset interpretation notes

To be written in clean Notebook 03:

- **Temperature** is the strongest cross-dataset signal: MetroPT-3 `Oil_temperature` shows a consistent pre-failure direction at 6h (but limited formal power), while the hydraulic `TS1_mean` / `TS1_std` results for `cooler_condition` are the strongest formal findings in the project.
- **Pressure** is only partially supported: MetroPT-3 `TP2` shows a 2-of-3 directional pattern but weak event-level confirmation, while hydraulic pressure-related features play mainly a secondary role rather than acting as dominant discriminators.
- **Electrical / load-related behavior** is the most divergent concept: MetroPT-3 `Motor_current` was excluded from formal testing because of inconsistent direction, while hydraulic `EPS1_mean` provides a robust signal for `pump_leakage`.
- `valve_condition` and `accumulator_pressure` are excluded from the cross-dataset comparison because Notebook 02 showed weak feature separation and no clear MetroPT-3 analogue was retained for formal comparison.
- **Hypothesis 2 verdict:** partially supported. Concordance is strongest for temperature, partial for pressure, and weak/divergent for electrical or load-related behavior.
- **Hypothesis 3 reliability context:** supported mainly through MetroPT-3. The hydraulic dataset contributes condition-state context rather than directly comparable MTBF / MTTR metrics.
- **Project synthesis:** the hydraulic dataset shows which features *can* discriminate degradation under controlled conditions, while MetroPT-3 shows which broad sensing concepts *remain visible* under real field operation.

## Notebook 03 closing notes

To be written in clean Notebook 03. Final summary:

- H1: supported strongly in the hydraulic dataset; weak but directionally meaningful in MetroPT-3
- H2: partially supported, strongest for temperature
- H3: supported mainly by MetroPT-3 reliability context, only indirectly on the hydraulic side
- Main methodological conclusion: degradation signals depend strongly on data structure (event-based field monitoring vs labeled cycle-based experiment)